In [1]:
# Setup
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !rm -rf /content/kg-bayesian-prior
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
    !pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn
    os.chdir('/content/kg-bayesian-prior')
    sys.path.insert(0, '/content/kg-bayesian-prior')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

Cloning into '/content/kg-bayesian-prior'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 180 (delta 99), reused 126 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 165.20 KiB | 3.30 MiB/s, done.
Resolving deltas: 100% (99/99), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.6/280.6 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.3/176.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.3 MB/s eta 0:00:00


In [2]:
import gc, json, warnings
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm

from src.data import load_fb15k237
from src.models.ggpn import GGPN
from src.utils.training import set_seed
from src.evaluation.calibration import expected_calibration_error, brier_score
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

warnings.filterwarnings('ignore')
device = "cuda" if torch.cuda.is_available() else "cpu"

# GPU info
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU available")

train_data, _, test_data = load_fb15k237()
print(f"Data: {len(train_data):,} train, {len(test_data):,} test")

GPU: NVIDIA A100-SXM4-40GB
Memory: 42.5 GB
FB15k-237 not found. Downloading...


train.txt: 21.0MB [00:00, 37.9MB/s]


valid.txt: 1.29MB [00:00, 3.36MB/s]


test.txt: 1.51MB [00:00, 6.01MB/s]


FB15k-237 download complete!
Data: 272,115 train, 20,466 test


In [3]:
def train_and_evaluate(seed):
    print(f"\n{'='*50}")
    print(f"SEED {seed}")
    print(f"{'='*50}")
    set_seed(seed)
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    # Training (100 dim)
    model = GGPN(train_data.num_entities, train_data.num_relations*2,
                 embedding_dim=100, hidden_dim=100, num_layers=1, num_rff=20).to(device)
    model.set_graph(train_data)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)

    for ep in (pbar := tqdm(range(50), desc=f"GGPN (seed={seed})")):
        model.train()
        loss_sum, n = 0, 0
        for st in range(0, len(train_data), 512):
            pos = torch.tensor(train_data.triples[st:st+512], device=device)
            neg = pos.clone()
            neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
            opt.zero_grad()
            loss = model.loss(pos, neg)
            loss = loss['total'] if isinstance(loss, dict) else loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            loss_sum += loss.item()
            n += 1
        pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

    # Evaluation
    model.eval()

    # MRR
    ranks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(test_data), 200), desc="MRR", leave=False):
            batch = test_data.triples[i:i+200]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            all_e = torch.arange(train_data.num_entities, device=device)
            scores = torch.stack([model(h[j].expand(train_data.num_entities),
                                        r[j].expand(train_data.num_entities), all_e) for j in range(len(h))])
            target = scores[torch.arange(len(t), device=device), t]
            ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())

    ranks = torch.tensor(ranks, dtype=torch.float)
    mrr = (1/ranks).mean().item()
    h1 = (ranks <= 1).float().mean().item()
    h10 = (ranks <= 10).float().mean().item()

    # ECE
    pos = test_data.triples
    neg = np.array([[h, r, np.random.randint(train_data.num_entities)] for h,r,t in pos])
    all_t = np.vstack([pos, neg])
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

    confs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(all_t), 1024), desc="ECE", leave=False):
            batch = all_t[i:i+1024]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            scores = model(h, r, t)
            confs.append(torch.sigmoid(scores).cpu().numpy())
    conf = np.concatenate(confs)
    ece, _ = expected_calibration_error(conf, labels)
    brier = brier_score(conf, labels)

    # AUROC
    ood_t = create_ood_dataset(train_data, test_data, "random", len(test_data))

    def get_unc(triples):
        uncs = []
        with torch.no_grad():
            for i in range(0, len(triples), 1024):
                batch = triples[i:i+1024]
                h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
                if hasattr(model, 'predict_with_uncertainty'):
                    pred = model.predict_with_uncertainty(h, r, t)
                    unc = pred.get('total', pred.get('epistemic'))
                else:
                    s = model(h, r, t)
                    p = torch.sigmoid(s)
                    unc = -p * torch.log(p + 1e-10) - (1-p) * torch.log(1-p + 1e-10)
                uncs.append(unc.cpu().numpy())
        return np.concatenate(uncs)

    auroc = compute_auroc(get_unc(test_data.triples), get_unc(ood_t))

    result = {"mrr": mrr, "hits@1": h1, "hits@10": h10, "ece": ece, "brier": brier, "auroc": auroc}
    print(f"Result: MRR={mrr:.4f}, H@1={h1:.4f}, H@10={h10:.4f}, ECE={ece:.4f}, AUROC={auroc:.4f}")

    del model
    return result

In [ ]:
# Run multiple seeds
SEEDS = [42, 123, 456]
all_results = {}

for seed in SEEDS:
    all_results[seed] = train_and_evaluate(seed)


SEED 42


GGPN (seed=42):   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
# Aggregate results
metrics = ['mrr', 'hits@1', 'hits@10', 'ece', 'brier', 'auroc']

print("\n" + "="*70)
print("GGPN RESULTS (100 dim, mean ± std)")
print("="*70)

summary = {}
for m in metrics:
    values = [all_results[s][m] for s in SEEDS]
    mean, std = np.mean(values), np.std(values)
    summary[m] = {"mean": mean, "std": std, "values": values}
    print(f"{m:>10}: {mean:.4f} ± {std:.4f}")

# Save to results folder
output = {"seeds": SEEDS, "results": all_results, "summary": summary}
result_path = "results/ggpn_results.json"
with open(result_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f"\nSaved to {result_path}")

# Push to GitHub (Colab only)
if IN_COLAB:
    !git config user.email "colab@experiment.com"
    !git config user.name "Colab Experiment"
    !git add results/ggpn_results.json
    !git commit -m "Add GGPN experiment results"
    !git push
    print("Pushed to GitHub!")